<div style="border: 5px dotted #2980B9; padding: 10px; background-color: #F0F0F0; border-radius: 10px; display: block; width: 100%; text-align: center; font-size: 2em;">
    <strong><span style="color:#FF6B6B"> Brain-to-Text '25 </span> || <span style="color:#9B59B6">An Introductory</span> <span style="color:#2980B9"> EDA</span></strong>
</div>

------
### Competition link: [Brain-to-text '25](https://www.kaggle.com/competitions/brain-to-text-25/overview)

-----


## Vocabulary Size

****A Quick Note on the Vocabulary****: The text hasn't been filtered for sensitive content. You may encounter words that are profane or offensive.

In [2]:
# data_augmentations.py

import torch
import torch.nn.functional as F
import numpy as np
from scipy.ndimage import gaussian_filter1d

def gauss_smooth(inputs, device, smooth_kernel_std=2, smooth_kernel_size=100, padding='same'):
    """
    Applies a 1D Gaussian smoothing operation with PyTorch to smooth the data along the time axis.
    Args:
        inputs (tensor : B x T x N): A 3D tensor with batch size B, time steps T, and number of features N.
                                     Assumed to already be on the correct device (e.g., GPU).
        smooth_kernel_std (float): Standard deviation of the Gaussian smoothing kernel.
        smooth_kernel_size (int): Size of the kernel used to generate the Gaussian.
        padding (str): Padding mode, either 'same' or 'valid'.
        device (str or torch.device): Device to use for computation (e.g., 'cuda' or 'cpu').
    Returns:
        smoothed (tensor : B x T x N): A smoothed 3D tensor with batch size B, time steps T, and number of features N.
    """
    inp = np.zeros(smooth_kernel_size, dtype=np.float32)
    inp[smooth_kernel_size // 2] = 1
    gaussKernel = gaussian_filter1d(inp, smooth_kernel_std)
    validIdx = np.argwhere(gaussKernel > 0.01)
    gaussKernel = gaussKernel[validIdx]
    gaussKernel = np.squeeze(gaussKernel / np.sum(gaussKernel))

    gaussKernel = torch.tensor(gaussKernel, dtype=torch.float32, device=device)
    gaussKernel = gaussKernel.view(1, 1, -1)

    B, T, C = inputs.shape
    inputs = inputs.permute(0, 2, 1)
    gaussKernel = gaussKernel.repeat(C, 1, 1)

    smoothed = F.conv1d(inputs, gaussKernel, padding=padding, groups=C)
    return smoothed.permute(0, 2, 1)


In [3]:
# dataset.py

import os
import torch
from torch.utils.data import Dataset
import h5py
import numpy as np
from torch.nn.utils.rnn import pad_sequence
import math

class BrainToTextDataset(Dataset):
    '''
    Dataset for brain-to-text data

    Returns an entire batch of data instead of a single example
    '''

    def __init__(
            self,
            trial_indicies,
            n_batches,
            split='train',
            batch_size=64,
            days_per_batch=1,
            random_seed=-1,
            must_include_days=None,
            feature_subset=None
    ):

        if random_seed != -1:
            np.random.seed(random_seed)
            torch.manual_seed(random_seed)

        self.split = split

        if self.split not in ['train', 'test']:
            raise ValueError(f'split must be either "train" or "test". Received {self.split}')

        self.days_per_batch = days_per_batch
        self.batch_size = batch_size
        self.n_batches = n_batches

        self.days = {}
        self.n_trials = 0
        self.trial_indicies = trial_indicies
        self.n_days = len(trial_indicies.keys())
        self.feature_subset = feature_subset

        for d in trial_indicies:
            self.n_trials += len(trial_indicies[d]['trials'])

        if must_include_days is not None and len(must_include_days) > days_per_batch:
            raise ValueError(f'must_include_days must be <= days_per_batch. '
                             f'Received {must_include_days}, days_per_batch {days_per_batch}')

        if must_include_days is not None and len(must_include_days) > self.n_days and split != 'train':
            raise ValueError(f'must_include_days is not valid for test data. '
                             f'Received {must_include_days} but only {self.n_days} in the dataset')

        if must_include_days is not None:
            for i, d in enumerate(must_include_days):
                if d < 0:
                    must_include_days[i] = self.n_days + d

        self.must_include_days = must_include_days

        if self.split == 'train' and self.days_per_batch > self.n_days:
            raise ValueError(f'Requested days_per_batch: {days_per_batch} is greater than available days {self.n_days}.')

        if self.split == 'train':
            self.batch_index = self.create_batch_index_train()
        else:
            self.batch_index = self.create_batch_index_test()
            self.n_batches = len(self.batch_index.keys())

    def __len__(self):
        '''
        How many batches are in this dataset.
        '''
        return self.n_batches

    def __getitem__(self, idx):
        '''
        Gets an entire batch of data from the dataset, not just a single item
        '''
        batch = {
            'input_features': [],
            'seq_class_ids': [],
            'n_time_steps': [],
            'phone_seq_lens': [],
            'day_indicies': [],
            'transcriptions': [],
            'block_nums': [],
            'trial_nums': [],
        }

        index = self.batch_index[idx]

        for d in index.keys():

            with h5py.File(self.trial_indicies[d]['session_path'], 'r') as f:

                for t in index[d]:
                    try:
                        g = f[f'trial_{t:04d}']

                        input_features = torch.from_numpy(g['input_features'][:])
                        if self.feature_subset:
                            input_features = input_features[:, self.feature_subset]

                        batch['input_features'].append(input_features)
                        batch['seq_class_ids'].append(torch.from_numpy(g['seq_class_ids'][:]))
                        batch['transcriptions'].append(torch.from_numpy(g['transcription'][:]))
                        batch['n_time_steps'].append(g.attrs['n_time_steps'])
                        batch['phone_seq_lens'].append(g.attrs['seq_len'])
                        batch['day_indicies'].append(int(d))
                        batch['block_nums'].append(g.attrs['block_num'])
                        batch['trial_nums'].append(g.attrs['trial_num'])

                    except Exception as e:
                        print(f'Error loading trial {t} from session {self.trial_indicies[d]["session_path"]}: {e}')
                        continue

        batch['input_features'] = pad_sequence(batch['input_features'], batch_first=True, padding_value=0)
        batch['seq_class_ids'] = pad_sequence(batch['seq_class_ids'], batch_first=True, padding_value=0)

        batch['n_time_steps'] = torch.tensor(batch['n_time_steps'])
        batch['phone_seq_lens'] = torch.tensor(batch['phone_seq_lens'])
        batch['day_indicies'] = torch.tensor(batch['day_indicies'])
        batch['transcriptions'] = torch.stack(batch['transcriptions'])
        batch['block_nums'] = torch.tensor(batch['block_nums'])
        batch['trial_nums'] = torch.tensor(batch['trial_nums'])

        return batch

    def create_batch_index_train(self):
        '''
        Create an index that maps a batch_number to batch_size number of trials
        '''
        batch_index = {}

        if self.must_include_days is not None:
            non_must_include_days = [d for d in self.trial_indicies.keys() if d not in self.must_include_days]

        for batch_idx in range(self.n_batches):
            batch = {}

            if self.must_include_days is not None and len(self.must_include_days) > 0:
                days = np.concatenate(
                    (self.must_include_days,
                     np.random.choice(non_must_include_days,
                                      size=self.days_per_batch - len(self.must_include_days),
                                      replace=False))
                )
            else:
                days = np.random.choice(list(self.trial_indicies.keys()), size=self.days_per_batch, replace=False)

            num_trials = math.ceil(self.batch_size / self.days_per_batch)

            for d in days:
                trial_idxs = np.random.choice(self.trial_indicies[d]['trials'], size=num_trials, replace=True)
                batch[d] = trial_idxs

            extra_trials = (num_trials * len(days)) - self.batch_size
            while extra_trials > 0:
                d = np.random.choice(days)
                batch[d] = batch[d][:-1]
                extra_trials -= 1

            batch_index[batch_idx] = batch

        return batch_index

    def create_batch_index_test(self):
        '''
        Create an index that is all validation/testing data in batches of up to self.batch_size
        '''
        batch_index = {}
        batch_idx = 0

        for d in self.trial_indicies.keys():
            num_trials = len(self.trial_indicies[d]['trials'])
            num_batches = (num_trials + self.batch_size - 1) // self.batch_size

            for i in range(num_batches):
                start_idx = i * self.batch_size
                end_idx = min((i + 1) * self.batch_size, num_trials)
                batch_trials = self.trial_indicies[d]['trials'][start_idx:end_idx]
                batch_index[batch_idx] = {d: batch_trials}
                batch_idx += 1

        return batch_index


def train_test_split_indicies(file_paths, test_percentage=0.1, seed=-1, bad_trials_dict=None):
    '''
    Split data from file_paths into train and test splits
    Returns two dictionaries that detail which trials in each day will be a part of that split
    '''
    if seed != -1:
        np.random.seed(seed)

    trials_per_day = {}
    for i, path in enumerate(file_paths):
        session = [s for s in path.split('/') if (s.startswith('t15.20') or s.startswith('t12.20'))][0]
        good_trial_indices = []

        if os.path.exists(path):
            with h5py.File(path, 'r') as f:
                num_trials = len(list(f.keys()))
                for t in range(num_trials):
                    key = f'trial_{t:04d}'
                    block_num = f[key].attrs['block_num']
                    trial_num = f[key].attrs['trial_num']

                    if (
                        bad_trials_dict is not None
                        and session in bad_trials_dict
                        and str(block_num) in bad_trials_dict[session]
                        and trial_num in bad_trials_dict[session][str(block_num)]
                    ):
                        continue

                    good_trial_indices.append(t)

        trials_per_day[i] = {
            'num_trials': len(good_trial_indices),
            'trial_indices': good_trial_indices,
            'session_path': path
        }

    train_trials = {}
    test_trials = {}

    for day in trials_per_day.keys():
        num_trials = trials_per_day[day]['num_trials']
        all_trial_indices = trials_per_day[day]['trial_indices']
        session_path = trials_per_day[day]['session_path']

        if test_percentage == 0:
            train_trials[day] = {'trials': all_trial_indices, 'session_path': session_path}
            test_trials[day] = {'trials': [], 'session_path': session_path}
            continue
        elif test_percentage == 1:
            train_trials[day] = {'trials': [], 'session_path': session_path}
            test_trials[day] = {'trials': all_trial_indices, 'session_path': session_path}
            continue

        num_test = max(1, int(num_trials * test_percentage))
        test_indices = np.random.choice(all_trial_indices, size=num_test, replace=False).tolist()
        train_indices = [idx for idx in all_trial_indices if idx not in test_indices]

        train_trials[day] = {'trials': train_indices, 'session_path': session_path}
        test_trials[day] = {'trials': test_indices, 'session_path': session_path}

    return train_trials, test_trials


In [14]:
import os
from omegaconf import OmegaConf

args_path = "/kaggle/input/optimal-args/rnn_args.yaml"
args = OmegaConf.load(args_path)

print("Loaded config from:", args_path)

args['dataset']['dataset_dir'] = (
    "/kaggle/input/brain-to-text-25-data/t15_copyTask_neuralData/hdf5_data_final"
)

args['output_dir'] = "/kaggle/working/trained_models/baseline_rnn"
args['checkpoint_dir'] = os.path.join(args['output_dir'], "checkpoint")

os.makedirs(args['output_dir'], exist_ok=True)
os.makedirs(args['checkpoint_dir'], exist_ok=True)

args['mode'] = 'train'
args['gpu_number'] = '0'
# Transformer Parameters
args['dataset']['batch_size'] = 8
args['dataset']['num_dataloader_workers'] = 2
args['save_val_logits'] = False
args['save_val_data'] = False
args['use_torch_compile'] = False
args['use_amp'] = False

# Parameters for GRU Model
#args['model']['n_units'] = 384
#args['model']['n_layers'] = 3
#args['model']['patch_size'] = 0
#args['model']['patch_stride'] = 0
#args['dataset']['batch_size'] = 16
#args['dataset']['days_per_batch'] = 2

# args['num_training_batches'] = 40000
#args['use_compile'] = False

#print("Final args summary:")
#print("n_units:", args['model']['n_units'])
#print("n_layers:", args['model']['n_layers'])
#print("patch_size:", args['model']['patch_size'])
#print("batch_size:", args['dataset']['batch_size'])
#print("days_per_batch:", args['dataset']['days_per_batch'])

#print(args['dataset']['dataset_dir'])
#print(args['output_dir'])
#print(args['checkpoint_dir'])


Loaded config from: /kaggle/input/optimal-args/rnn_args.yaml


In [15]:
import os
import math
import json
import random
import time
import logging
import pathlib
import sys
import pickle

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LambdaLR

from omegaconf import OmegaConf

import torchaudio.functional as taF

torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic = True
torch._dynamo.config.cache_size_limit = 64



In [16]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    """
    Standard sine-cosine positional encoding for sequences.
    Produces [B, T, d_model] same shape as input, added elementwise.
    """
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x):
        """
        x: [B, T, d_model]
        """
        T = x.size(1)
        return x + self.pe[:, :T, :]


class TransformerDecoder(nn.Module):
    """
    Transformer version of the Brain-to-Text decoder.

    It mirrors GRUDecoder:
    - day-specific linear layers
    - optional patching over time
    - CTC-compatible logits: [B, T', n_classes]
    - forward(x, day_idx, states=None, return_state=False)
    """

    def __init__(
        self,
        neural_dim,
        n_units,
        n_days,
        n_classes,
        rnn_dropout=0.0,
        input_dropout=0.0,
        n_layers=4,
        patch_size=0,
        patch_stride=0,
        n_heads=8,
        dim_feedforward=None,
    ):
        super().__init__()

        self.neural_dim = neural_dim
        self.n_units = n_units
        self.n_classes = n_classes
        self.n_layers = n_layers
        self.n_days = n_days

        self.rnn_dropout = rnn_dropout
        self.input_dropout = input_dropout
        self.patch_size = patch_size
        self.patch_stride = patch_stride

        self.day_layer_activation = nn.Softsign()

        self.day_weights = nn.ParameterList(
            [nn.Parameter(torch.eye(self.neural_dim)) for _ in range(self.n_days)]
        )
        self.day_biases = nn.ParameterList(
            [nn.Parameter(torch.zeros(1, self.neural_dim)) for _ in range(self.n_days)]
        )

        self.day_layer_dropout = nn.Dropout(input_dropout)

        # Input size before projection
        self.input_size = self.neural_dim
        if self.patch_size > 0:
            self.input_size *= self.patch_size

        # Project to transformer
        self.input_proj = nn.Linear(self.input_size, self.n_units)

        # Positional encoding
        self.pos_encoding = PositionalEncoding(self.n_units)

        # Transformer Encoder
        if dim_feedforward is None:
            dim_feedforward = 4 * self.n_units

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.n_units,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=self.rnn_dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=self.n_layers,
        )

        self.out = nn.Linear(self.n_units, self.n_classes)
        nn.init.xavier_uniform_(self.out.weight)

    def _apply_day_layers(self, x, day_idx):
        """
        x: [B, T, D]
        day_idx: [B] (int day index per trial)
        """
        day_weights = torch.stack([self.day_weights[i] for i in day_idx], dim=0)
        day_biases = torch.cat([self.day_biases[i] for i in day_idx], dim=0).unsqueeze(1)

        x = torch.einsum("btd,bdk->btk", x, day_weights) + day_biases
        x = self.day_layer_activation(x)

        if self.input_dropout > 0:
            x = self.day_layer_dropout(x)

        return x

    def _apply_patching(self, x):
        """
        Optional patching over time, copied from GRUDecoder logic.

        x: [B, T, D]
        returns: [B, T_patch, D * patch_size] if patching enabled,
                 else [B, T, D]
        """
        if self.patch_size <= 0:
            return x

        x = x.unsqueeze(1)
        x = x.permute(0, 3, 1, 2)
        x_unfold = x.unfold(3, self.patch_size, self.patch_stride)
        x_unfold = x_unfold.squeeze(2)
        x_unfold = x_unfold.permute(0, 2, 3, 1)
        x = x_unfold.reshape(x.size(0), x_unfold.size(1), -1)
        return x

    def forward(self, x, day_idx, states=None, return_state=False):
        """
        x: [B, T, neural_dim]
        day_idx: [B] int indices of day

        returns:
          logits: [B, T', n_classes]
          (optionally) None as "hidden state", to match GRUDecoder interface
        """
        # Day specific normalization
        x = self._apply_day_layers(x, day_idx)

        x = self._apply_patching(x)

        # Project to transformer d_model
        x = self.input_proj(x)

        # positional encoding
        x = self.pos_encoding(x)
        x = self.transformer(x)
        # Prediction head
        logits = self.out(x)

        if return_state:
            return logits, None
        return logits


In [7]:
# rnn_model.py

import torch
from torch import nn

class GRUDecoder(nn.Module):
    '''
    Defines the GRU decoder

    This class combines day-specific input layers, a GRU, and an output classification layer
    '''
    def __init__(self,
                 neural_dim,
                 n_units,
                 n_days,
                 n_classes,
                 rnn_dropout=0.0,
                 input_dropout=0.0,
                 n_layers=5,
                 patch_size=0,
                 patch_stride=0,
                 ):
        super(GRUDecoder, self).__init__()

        self.neural_dim = neural_dim
        self.n_units = n_units
        self.n_classes = n_classes
        self.n_layers = n_layers
        self.n_days = n_days

        self.rnn_dropout = rnn_dropout
        self.input_dropout = input_dropout

        self.patch_size = patch_size
        self.patch_stride = patch_stride

        self.day_layer_activation = nn.Softsign()

        self.day_weights = nn.ParameterList(
            [nn.Parameter(torch.eye(self.neural_dim)) for _ in range(self.n_days)]
        )
        self.day_biases = nn.ParameterList(
            [nn.Parameter(torch.zeros(1, self.neural_dim)) for _ in range(self.n_days)]
        )

        self.day_layer_dropout = nn.Dropout(input_dropout)
        self.input_size = self.neural_dim

        if self.patch_size > 0:
            self.input_size *= self.patch_size

        self.gru = nn.GRU(
            input_size=self.input_size,
            hidden_size=self.n_units,
            num_layers=self.n_layers,
            dropout=self.rnn_dropout,
            batch_first=True,
            bidirectional=False,
        )

        for name, param in self.gru.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            if "weight_ih" in name:
                nn.init.xavier_uniform_(param)

        self.out = nn.Linear(self.n_units, self.n_classes)
        nn.init.xavier_uniform_(self.out.weight)

        self.h0 = nn.Parameter(nn.init.xavier_uniform_(torch.zeros(1, 1, self.n_units)))

    def forward(self, x, day_idx, states=None, return_state=False):
        '''
        x        (tensor)  - [B, T, D]
        day_idx  (tensor)  - [B] day indices
        '''
        day_weights = torch.stack([self.day_weights[i] for i in day_idx], dim=0)
        day_biases = torch.cat([self.day_biases[i] for i in day_idx], dim=0).unsqueeze(1)

        x = torch.einsum("btd,bdk->btk", x, day_weights) + day_biases
        x = self.day_layer_activation(x)

        if self.input_dropout > 0:
            x = self.day_layer_dropout(x)

        if self.patch_size > 0:
            x = x.unsqueeze(1)
            x = x.permute(0, 3, 1, 2)
            x_unfold = x.unfold(3, self.patch_size, self.patch_stride)
            x_unfold = x_unfold.squeeze(2)
            x_unfold = x_unfold.permute(0, 2, 3, 1)
            x = x_unfold.reshape(x.size(0), x_unfold.size(1), -1)

        if states is None:
            states = self.h0.expand(self.n_layers, x.shape[0], self.n_units).contiguous()

        output, hidden_states = self.gru(x, states)
        logits = self.out(output)

        if return_state:
            return logits, hidden_states
        return logits


In [17]:
class BrainToTextDecoder_Trainer:
    """
    Initialize and train the brain-to-text phoneme decoder (baseline RNN)
    Adapted from original project, but made notebook/Kaggle friendly.
    """

    def __init__(self, args):
        self.args = args
        self.logger = None 
        self.device = None
        self.model = None
        self.optimizer = None
        self.learning_rate_scheduler = None
        self.ctc_loss = None 

        self.best_val_PER = torch.inf
        self.best_val_loss = torch.inf

        self.train_dataset = None 
        self.val_dataset = None 
        self.train_loader = None 
        self.val_loader = None 

        self.transform_args = self.args['dataset']['data_transforms']

        if args['mode'] == 'train':
            os.makedirs(self.args['output_dir'], exist_ok=True)
        if args['save_best_checkpoint'] or args['save_all_val_steps'] or args['save_final_model']:
            os.makedirs(self.args['checkpoint_dir'], exist_ok=True)

        self.logger = logging.getLogger("BrainToTextTrainer")
        for h in list(self.logger.handlers):
            self.logger.removeHandler(h)
        self.logger.setLevel(logging.INFO)
        formatter = logging.Formatter(fmt='%(asctime)s: %(message)s')

        if args['mode'] == 'train':
            fh = logging.FileHandler(str(pathlib.Path(self.args['output_dir'], 'training_log')))
            fh.setFormatter(formatter)
            self.logger.addHandler(fh)

        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(formatter)
        self.logger.addHandler(sh)

        if torch.cuda.is_available():
            gpu_num = self.args.get('gpu_number', 0)
            try:
                gpu_num = int(gpu_num)
            except ValueError:
                self.logger.warning(f"Invalid gpu_number value: {gpu_num}. Using 0 instead.")
                gpu_num = 0

            max_gpu_index = torch.cuda.device_count() - 1
            if gpu_num > max_gpu_index:
                self.logger.warning(f"Requested GPU {gpu_num} not available. Using GPU 0 instead.")
                gpu_num = 0

            try:
                self.device = torch.device(f"cuda:{gpu_num}")
                _ = torch.tensor([1.0]).to(self.device) * 2
            except Exception as e:
                self.logger.error(f"Error initializing CUDA device {gpu_num}: {str(e)}")
                self.logger.info("Falling back to CPU")
                self.device = torch.device("cpu")
        else:
            self.device = torch.device("cpu")

        self.logger.info(f'Using device: {self.device}')

        if self.args['seed'] != -1:
            np.random.seed(self.args['seed'])
            random.seed(self.args['seed'])
            torch.manual_seed(self.args['seed'])

        # Model
        hidden_size = 256
        num_layers  = 2
        num_heads   = 4
        ff_dim      = 1024  

        self.args['model']['n_units']       = hidden_size
        self.args['model']['n_layers']      = num_layers
        self.args['model']['patch_size']    = 0      
        self.args['model']['patch_stride']  = 0

        self.args['model']['rnn_dropout'] = 0.1

        self.model = TransformerDecoder(
            neural_dim   = self.args['model']['n_input_features'],
            n_units      = hidden_size,
            n_days       = len(self.args['dataset']['sessions']),
            n_classes    = self.args['dataset']['n_classes'],
            rnn_dropout  = self.args['model']['rnn_dropout'],
            input_dropout= self.args['model']['input_network']['input_layer_dropout'],
            n_layers     = num_layers,
            patch_size   = 0,
            patch_stride = 0,
            n_heads      = num_heads,
            dim_feedforward = ff_dim,
        )


        if self.args['use_torch_compile']:
            self.model = torch.compile(self.model)

        self.logger.info("Initialized RNN decoding model")
        self.logger.info(self.model)

        total_params = sum(p.numel() for p in self.model.parameters())
        self.logger.info(f"Model has {total_params:,} parameters")

        day_params = 0
        for name, param in self.model.named_parameters():
            if 'day' in name:
                day_params += param.numel()
        self.logger.info(
            f"Model has {day_params:,} day-specific parameters "
            f"| {((day_params / total_params) * 100):.2f}% of total parameters"
        )

        # Datasets and splits
        train_file_paths = [
            os.path.join(self.args["dataset"]["dataset_dir"], s, 'data_train.hdf5')
            for s in self.args['dataset']['sessions']
        ]
        val_file_paths = [
            os.path.join(self.args["dataset"]["dataset_dir"], s, 'data_val.hdf5')
            for s in self.args['dataset']['sessions']
        ]

        if len(set(train_file_paths)) != len(train_file_paths):
            raise ValueError("Duplicate sessions listed in the train dataset")
        if len(set(val_file_paths)) != len(val_file_paths):
            raise ValueError("Duplicate sessions listed in the val dataset")

        train_trials, _ = train_test_split_indicies(
            file_paths = train_file_paths, 
            test_percentage = 0,
            seed = self.args['dataset']['seed'],
            bad_trials_dict = self.args['dataset'].get('bad_trials_dict', None),
        )
        _, val_trials = train_test_split_indicies(
            file_paths = val_file_paths, 
            test_percentage = 1,
            seed = self.args['dataset']['seed'],
            bad_trials_dict = self.args['dataset'].get('bad_trials_dict', None),
        )

        with open(os.path.join(self.args['output_dir'], 'train_val_trials.json'), 'w') as f:
            json.dump({'train': train_trials, 'val': val_trials}, f)

        feature_subset = None
        if ('feature_subset' in self.args['dataset']) and self.args['dataset']['feature_subset'] is not None:
            feature_subset = self.args['dataset']['feature_subset']
            self.logger.info(f'Using only a subset of features: {feature_subset}')

        self.train_dataset = BrainToTextDataset(
            trial_indicies = train_trials,
            split = 'train',
            days_per_batch = self.args['dataset']['days_per_batch'],
            n_batches = self.args['num_training_batches'],
            batch_size = self.args['dataset']['batch_size'],
            must_include_days = None,
            random_seed = self.args['dataset']['seed'],
            feature_subset = feature_subset
        )
        self.train_loader = DataLoader(
            self.train_dataset,
            batch_size = None,
            shuffle = self.args['dataset']['loader_shuffle'],
            num_workers = self.args['dataset']['num_dataloader_workers'],
            pin_memory = True,
        )

        self.val_dataset = BrainToTextDataset(
            trial_indicies = val_trials, 
            split = 'test',
            days_per_batch = None,
            n_batches = None,
            batch_size = self.args['dataset']['batch_size'],
            must_include_days = None,
            random_seed = self.args['dataset']['seed'],
            feature_subset = feature_subset,
        )
        self.val_loader = DataLoader(
            self.val_dataset,
            batch_size = None,
            shuffle = False,
            num_workers = 0,
            pin_memory = True,
        )

        self.logger.info("Successfully initialized datasets")

        self.optimizer = self.create_optimizer()

        if self.args['lr_scheduler_type'] == 'linear':
            self.learning_rate_scheduler = torch.optim.lr_scheduler.LinearLR(
                optimizer = self.optimizer,
                start_factor = 1.0,
                end_factor = self.args['lr_min'] / self.args['lr_max'],
                total_iters = self.args['lr_decay_steps'],
            )
        elif self.args['lr_scheduler_type'] == 'cosine':
            self.learning_rate_scheduler = self.create_cosine_lr_scheduler(self.optimizer)
        else:
            raise ValueError(f"Invalid lr_scheduler_type: {self.args['lr_scheduler_type']}")

        self.ctc_loss = torch.nn.CTCLoss(blank=0, reduction='none', zero_infinity=False)

        if self.args['init_from_checkpoint'] and self.args['init_checkpoint_path']:
            self.load_model_checkpoint(self.args['init_checkpoint_path'])

        for name, param in self.model.named_parameters():
            if not self.args['model']['rnn_trainable'] and 'gru' in name:
                param.requires_grad = False
            elif not self.args['model']['input_network']['input_trainable'] and 'day' in name:
                param.requires_grad = False

        self.model.to(self.device)

    def create_optimizer(self):
        """
        Create the optimizer with special param groups.
        Biases and day-parameters should not be weight-decayed.
        Day parameters also get their own LR.
        """

        bias_params = []
        day_params = []
        other_params = []

        for name, p in self.model.named_parameters():
            if not p.requires_grad:
                continue

            if "gru.bias" in name or "out.bias" in name:
                bias_params.append(p)

            elif "day" in name:
                day_params.append(p)

            else:
                other_params.append(p)

        param_groups = []
        if bias_params:
            param_groups.append(
                {
                    "params": bias_params,
                    "weight_decay": 0.0,
                    "group_type": "bias",
                }
            )
        if day_params:
            param_groups.append(
                {
                    "params": day_params,
                    "lr": self.args["lr_max_day"],
                    "weight_decay": self.args["weight_decay_day"],
                    "group_type": "day_layer",
                }
            )
        if other_params:
            param_groups.append(
                {
                    "params": other_params,
                    "group_type": "other",
                }
            )

        try:
            optim = torch.optim.AdamW(
                param_groups,
                lr=self.args["lr_max"],
                betas=(self.args["beta0"], self.args["beta1"]),
                eps=self.args["epsilon"],
                weight_decay=self.args["weight_decay"],
                fused=True,
            )
        except TypeError:
            optim = torch.optim.AdamW(
                param_groups,
                lr=self.args["lr_max"],
                betas=(self.args["beta0"], self.args["beta1"]),
                eps=self.args["epsilon"],
                weight_decay=self.args["weight_decay"],
            )

        return optim


    def create_cosine_lr_scheduler(self, optim):
        lr_max = self.args['lr_max']
        lr_min = self.args['lr_min']
        lr_decay_steps = self.args['lr_decay_steps']

        lr_max_day =  self.args['lr_max_day']
        lr_min_day = self.args['lr_min_day']
        lr_decay_steps_day = self.args['lr_decay_steps_day']

        lr_warmup_steps = self.args['lr_warmup_steps']
        lr_warmup_steps_day = self.args['lr_warmup_steps_day']

        def lr_lambda(current_step, min_lr_ratio, decay_steps, warmup_steps):
            if current_step < warmup_steps:
                return float(current_step) / float(max(1, warmup_steps))
            if current_step < decay_steps:
                progress = float(current_step - warmup_steps) / float(
                    max(1, decay_steps - warmup_steps)
                )
                cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
                return max(min_lr_ratio, min_lr_ratio + (1 - min_lr_ratio) * cosine_decay)
            return min_lr_ratio

        if len(optim.param_groups) == 3:
            lr_lambdas = [
                lambda step: lr_lambda(step, lr_min / lr_max, lr_decay_steps, lr_warmup_steps),
                lambda step: lr_lambda(step, lr_min_day / lr_max_day, lr_decay_steps_day, lr_warmup_steps_day),
                lambda step: lr_lambda(step, lr_min / lr_max, lr_decay_steps, lr_warmup_steps),
            ]
        elif len(optim.param_groups) == 2:
            lr_lambdas = [
                lambda step: lr_lambda(step, lr_min / lr_max, lr_decay_steps, lr_warmup_steps),
                lambda step: lr_lambda(step, lr_min / lr_max, lr_decay_steps, lr_warmup_steps),
            ]
        else:
            raise ValueError(f"Unexpected number of param groups: {len(optim.param_groups)}")

        return LambdaLR(optim, lr_lambdas, -1)

    def load_model_checkpoint(self, load_path):
        checkpoint = torch.load(load_path, weights_only=False, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.learning_rate_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        self.best_val_PER = checkpoint.get('val_PER', torch.inf)
        self.best_val_loss = checkpoint.get('val_loss', torch.inf)

        self.model.to(self.device)
        for state in self.optimizer.state.values():
            for k, v in state.items():
                if isinstance(v, torch.Tensor):
                    state[k] = v.to(self.device)

        self.logger.info(f"Loaded model from checkpoint: {load_path}")

    def save_model_checkpoint(self, save_path, PER, loss):
        checkpoint = {
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.learning_rate_scheduler.state_dict(),
            'val_PER': PER,
            'val_loss': loss,
        }
        torch.save(checkpoint, save_path)
        self.logger.info(f"Saved model to checkpoint: {save_path}")

        args_save_path = os.path.join(self.args['checkpoint_dir'], 'args.yaml')
        OmegaConf.save(config=self.args, f=args_save_path)

    def transform_data(self, features, n_time_steps, mode='train'):
        data_shape = features.shape
        batch_size = data_shape[0]
        channels = data_shape[-1]

        if mode == 'train':
            if self.transform_args['static_gain_std'] > 0:
                warp_mat = torch.tile(torch.unsqueeze(torch.eye(channels, device=self.device), dim=0),
                                      (batch_size, 1, 1))
                warp_mat += torch.randn_like(warp_mat) * self.transform_args['static_gain_std']
                features = torch.matmul(features, warp_mat)

            if self.transform_args['white_noise_std'] > 0:
                features += torch.randn(data_shape, device=self.device) * self.transform_args['white_noise_std']

            if self.transform_args['constant_offset_std'] > 0:
                features += torch.randn((batch_size, 1, channels), device=self.device) * \
                            self.transform_args['constant_offset_std']

            if self.transform_args['random_walk_std'] > 0:
                features += torch.cumsum(
                    torch.randn(data_shape, device=self.device) * self.transform_args['random_walk_std'],
                    dim=self.transform_args['random_walk_axis'],
                )

            if self.transform_args['random_cut'] > 0:
                cut = np.random.randint(0, self.transform_args['random_cut'])
                features = features[:, cut:, :]
                n_time_steps = n_time_steps - cut

        if self.transform_args['smooth_data']:
            features = gauss_smooth(
                inputs=features,
                device=self.device,
                smooth_kernel_std=self.transform_args['smooth_kernel_std'],
                smooth_kernel_size=self.transform_args['smooth_kernel_size'],
            )

        return features, n_time_steps

    def train(self):
        self.model.train()

        train_losses = []
        val_losses = []
        val_PERs = []
        val_results = []

        val_steps_since_improvement = 0

        save_best_checkpoint = self.args.get('save_best_checkpoint', True)
        early_stopping = self.args.get('early_stopping', False)
        early_stopping_val_steps = self.args['early_stopping_val_steps']

        train_start_time = time.time()

        for i, batch in enumerate(self.train_loader):
            self.model.train()
            self.optimizer.zero_grad()

            start_time = time.time()

            features = batch['input_features'].to(self.device)
            labels = batch['seq_class_ids'].to(self.device)
            n_time_steps = batch['n_time_steps'].to(self.device)
            phone_seq_lens = batch['phone_seq_lens'].to(self.device)
            day_indicies = batch['day_indicies'].to(self.device)

            with torch.autocast(device_type="cuda", enabled=self.args['use_amp'], dtype=torch.bfloat16):
                features, n_time_steps = self.transform_data(features, n_time_steps, 'train')

                if self.args['model']['patch_size'] > 0:
                    adjusted_lens = ((n_time_steps - self.args['model']['patch_size']) /
                                     self.args['model']['patch_stride'] + 1).to(torch.int32)
                else:
                    adjusted_lens = n_time_steps.to(torch.int32)

                logits = self.model(features, day_indicies)

                loss = self.ctc_loss(
                    log_probs=torch.permute(logits.log_softmax(2), [1, 0, 2]),
                    targets=labels,
                    input_lengths=adjusted_lens,
                    target_lengths=phone_seq_lens,
                )
                loss = loss.mean()

            loss.backward()

            grad_norm = None
            if self.args['grad_norm_clip_value'] > 0:
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    max_norm=self.args['grad_norm_clip_value'],
                    error_if_nonfinite=True,
                    foreach=True,
                )

            self.optimizer.step()
            self.learning_rate_scheduler.step()

            train_step_duration = time.time() - start_time
            train_losses.append(loss.detach().item())

            if i % self.args['batches_per_train_log'] == 0:
                self.logger.info(
                    f'Train batch {i}: loss={loss.detach().item():.3f} '
                    f'grad_norm={grad_norm if grad_norm is not None else 0:.2f} '
                    f'time={train_step_duration:.3f}'
                )

            if i % self.args['batches_per_val_step'] == 0 or i == (self.args['num_training_batches'] - 1):
                self.logger.info(f"Running validation after training batch {i}")
                start_time = time.time()
                val_metrics = self.validation(
                    loader=self.val_loader,
                    return_logits=self.args['save_val_logits'],
                    return_data=self.args['save_val_data'],
                )
                val_step_duration = time.time() - start_time

                self.logger.info(
                    f'Val batch {i}: PER={val_metrics["avg_PER"]:.4f} '
                    f'CTC loss={val_metrics["avg_loss"]:.4f} time={val_step_duration:.3f}'
                )

                if self.args['log_individual_day_val_PER']:
                    for d in val_metrics['day_PERs'].keys():
                        self.logger.info(
                            f"{self.args['dataset']['sessions'][d]} val PER: "
                            f"{val_metrics['day_PERs'][d]['total_edit_distance'] / val_metrics['day_PERs'][d]['total_seq_length']:.4f}"
                        )

                val_PERs.append(val_metrics['avg_PER'])
                val_losses.append(val_metrics['avg_loss'])
                val_results.append(val_metrics)

                new_best = False
                if val_metrics['avg_PER'] < self.best_val_PER:
                    self.logger.info(
                        f"New best val PER {self.best_val_PER:.4f} → {val_metrics['avg_PER']:.4f}"
                    )
                    self.best_val_PER = val_metrics['avg_PER']
                    self.best_val_loss = val_metrics['avg_loss']
                    new_best = True
                elif val_metrics['avg_PER'] == self.best_val_PER and val_metrics['avg_loss'] < self.best_val_loss:
                    self.logger.info(
                        f"New best val loss {self.best_val_loss:.4f} → {val_metrics['avg_loss']:.4f}"
                    )
                    self.best_val_loss = val_metrics['avg_loss']
                    new_best = True

                if new_best:
                    if save_best_checkpoint:
                        self.logger.info("Checkpointing best model")
                        self.save_model_checkpoint(
                            os.path.join(self.args["checkpoint_dir"], 'best_checkpoint'),
                            self.best_val_PER,
                            self.best_val_loss,
                        )

                    if self.args['save_val_metrics']:
                        with open(os.path.join(self.args["checkpoint_dir"], 'val_metrics.pkl'), 'wb') as f:
                            pickle.dump(val_metrics, f)

                    val_steps_since_improvement = 0
                else:
                    val_steps_since_improvement += 1

                if self.args['save_all_val_steps']:
                    ckpt_path = os.path.join(self.args["checkpoint_dir"], f'checkpoint_batch_{i}')
                    self.save_model_checkpoint(ckpt_path, val_metrics['avg_PER'], val_metrics['avg_loss'])

                if early_stopping and (val_steps_since_improvement >= early_stopping_val_steps):
                    self.logger.info(
                        f'Val PER has not improved in {early_stopping_val_steps} validation steps. '
                        f'Stopping early at batch {i}.'
                    )
                    break

        training_duration = time.time() - train_start_time
        self.logger.info(f'Best avg val PER achieved: {self.best_val_PER:.5f}')
        self.logger.info(f'Total training time: {training_duration / 60:.2f} minutes')

        if self.args['save_final_model']:
            final_ckpt = os.path.join(
                self.args["checkpoint_dir"], f'final_checkpoint_batch_{i}')
            self.save_model_checkpoint(final_ckpt, val_PERs[-1], val_losses[-1])

        train_stats = {
            'train_losses': train_losses,
            'val_losses': val_losses,
            'val_PERs': val_PERs,
            'val_metrics': val_results,
        }
        return train_stats

    def validation(self, loader, return_logits=False, return_data=False):
        self.model.eval()
        metrics = {}

        if return_logits:
            metrics['logits'] = []
            metrics['n_time_steps'] = []

        if return_data:
            metrics['input_features'] = []

        metrics['decoded_seqs'] = []
        metrics['true_seq'] = []
        metrics['phone_seq_lens'] = []
        metrics['transcription'] = []
        metrics['losses'] = []
        metrics['block_nums'] = []
        metrics['trial_nums'] = []
        metrics['day_indicies'] = []

        total_edit_distance = 0
        total_seq_length = 0

        day_per = {}
        for d in range(len(self.args['dataset']['sessions'])):
            if self.args['dataset']['dataset_probability_val'][d] == 1:
                day_per[d] = {'total_edit_distance': 0, 'total_seq_length': 0}

        for i, batch in enumerate(loader):
            features = batch['input_features'].to(self.device)
            labels = batch['seq_class_ids'].to(self.device)
            n_time_steps = batch['n_time_steps'].to(self.device)
            phone_seq_lens = batch['phone_seq_lens'].to(self.device)
            day_indicies = batch['day_indicies'].to(self.device)

            day = day_indicies[0].item()
            if self.args['dataset']['dataset_probability_val'][day] == 0:
                continue

            with torch.no_grad():
                with torch.autocast(device_type="cuda", enabled=self.args['use_amp'], dtype=torch.bfloat16):
                    features, n_time_steps = self.transform_data(features, n_time_steps, 'val')

                    if self.args['model']['patch_size'] > 0:
                        adjusted_lens = ((n_time_steps - self.args['model']['patch_size']) /
                                         self.args['model']['patch_stride'] + 1).to(torch.int32)
                    else:
                        adjusted_lens = n_time_steps.to(torch.int32)

                    logits = self.model(features, day_indicies)

                    loss = self.ctc_loss(
                        torch.permute(logits.log_softmax(2), [1, 0, 2]),
                        labels,
                        adjusted_lens,
                        phone_seq_lens,
                    )
                    loss = torch.mean(loss)

                metrics['losses'].append(loss.cpu().detach().numpy())

                batch_edit_distance = 0
                decoded_seqs = []
                for b in range(logits.shape[0]):
                    T = adjusted_lens[b].item()
                    decoded_seq = torch.argmax(logits[b, :T, :], dim=-1)
                    decoded_seq = torch.unique_consecutive(decoded_seq, dim=-1)
                    decoded_seq = decoded_seq.cpu().detach().numpy()
                    decoded_seq = np.array([i for i in decoded_seq if i != 0])

                    true_seq = np.array(
                        labels[b][0:phone_seq_lens[b]].cpu().detach()
                    )
                    batch_edit_distance += taF.edit_distance(decoded_seq, true_seq)
                    decoded_seqs.append(decoded_seq)

            day_per[day]['total_edit_distance'] += batch_edit_distance
            day_per[day]['total_seq_length'] += torch.sum(phone_seq_lens).item()

            total_edit_distance += batch_edit_distance
            total_seq_length += torch.sum(phone_seq_lens)

            if return_logits:
                metrics['logits'].append(logits.cpu().float().numpy())
                metrics['n_time_steps'].append(adjusted_lens.cpu().numpy())

            if return_data:
                metrics['input_features'].append(batch['input_features'].cpu().numpy())

            metrics['decoded_seqs'].append(decoded_seqs)
            metrics['true_seq'].append(batch['seq_class_ids'].cpu().numpy())
            metrics['phone_seq_lens'].append(batch['phone_seq_lens'].cpu().numpy())
            metrics['transcription'].append(batch['transcriptions'].cpu().numpy())
            metrics['block_nums'].append(batch['block_nums'].numpy())
            metrics['trial_nums'].append(batch['trial_nums'].numpy())
            metrics['day_indicies'].append(batch['day_indicies'].cpu().numpy())

        avg_PER = (total_edit_distance / total_seq_length).item()

        metrics['day_PERs'] = day_per
        metrics['avg_PER'] = avg_PER
        metrics['avg_loss'] = float(np.mean(metrics['losses']))

        return metrics


In [18]:
# Run training

trainer = BrainToTextDecoder_Trainer(args)
train_stats = trainer.train()

print("Training done.")
print("Best val PER:", trainer.best_val_PER)


2025-11-14 03:23:04,159: Using device: cuda:0
2025-11-14 03:23:04,191: Initialized RNN decoding model
2025-11-14 03:23:04,192: TransformerDecoder(
  (day_layer_activation): Softsign()
  (day_weights): ParameterList(
      (0): Parameter containing: [torch.float32 of size 512x512]
      (1): Parameter containing: [torch.float32 of size 512x512]
      (2): Parameter containing: [torch.float32 of size 512x512]
      (3): Parameter containing: [torch.float32 of size 512x512]
      (4): Parameter containing: [torch.float32 of size 512x512]
      (5): Parameter containing: [torch.float32 of size 512x512]
      (6): Parameter containing: [torch.float32 of size 512x512]
      (7): Parameter containing: [torch.float32 of size 512x512]
      (8): Parameter containing: [torch.float32 of size 512x512]
      (9): Parameter containing: [torch.float32 of size 512x512]
      (10): Parameter containing: [torch.float32 of size 512x512]
      (11): Parameter containing: [torch.float32 of size 512x512]
  

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


2025-11-14 03:23:41,535: Successfully initialized datasets
2025-11-14 03:23:42,205: Train batch 0: loss=2254.374 grad_norm=13914.79 time=0.187
2025-11-14 03:23:42,207: Running validation after training batch 0
2025-11-14 03:24:26,427: Val batch 0: PER=7.0723 CTC loss=2137.9451 time=44.219
2025-11-14 03:24:26,428: t15.2023.08.13 val PER: 5.9470
2025-11-14 03:24:26,429: t15.2023.08.18 val PER: 6.7703
2025-11-14 03:24:26,430: t15.2023.08.20 val PER: 6.4178
2025-11-14 03:24:26,430: t15.2023.08.25 val PER: 6.4744
2025-11-14 03:24:26,431: t15.2023.08.27 val PER: 6.0096
2025-11-14 03:24:26,432: t15.2023.09.01 val PER: 6.6737
2025-11-14 03:24:26,432: t15.2023.09.03 val PER: 6.5273
2025-11-14 03:24:26,433: t15.2023.09.24 val PER: 7.7245
2025-11-14 03:24:26,434: t15.2023.09.29 val PER: 7.1366
2025-11-14 03:24:26,434: t15.2023.10.01 val PER: 5.7325
2025-11-14 03:24:26,435: t15.2023.10.06 val PER: 7.4327
2025-11-14 03:24:26,436: t15.2023.10.08 val PER: 5.3654
2025-11-14 03:24:26,436: t15.2023.10.1

KeyboardInterrupt: 